# Top-1,000 origin fact-check

**Status as of 5 September 2026:** All 1,000 rows were compared with the linked MusicBrainz records and available English Wikipedia artist-infobox origins. The current catalogue has 56 manual decisions, including 18 corrections and 11 resolutions that are already reflected in its origin clusters. Another 330 records remain proposed, source-variant, unresolved or contested. Automated corroboration does not establish that every formation place is correct.

The [full dataset review](top1000_full_review_20260905.ipynb) records the current quality findings and reproducible checks.

## Scope, data, and definitions

The population is the frozen 1,000-row catalogue `popularity_first_top1000_20260718T204522Z_bands.csv`. A **current claim** is the catalogue's formation label or an explicit reviewed override. External checks use MusicBrainz `begin-area` and English Wikipedia's artist-infobox `origin`. A match means compatible place wording after normalization; it does not establish that a founder's birthplace, a later base or a nearby major city is the formation place.

The automated comparison covers all 1,000 rows, using external evidence where available. The decision ledger supplies 56 manual decisions for this catalogue; its additional decision for Another Sunny Day concerns an artist outside the current selection. Manual decisions use the row-level sources recorded in the ledger. The remainder of the catalogue has not received independent manual adjudication in this audit.

In [1]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'requirements.txt').exists() and (p / 'data').exists())
audit = pd.read_csv(root / 'data/processed/top1000_origin_fact_check_20260901.csv', keep_default_na=False)
report = json.loads((root / 'data/processed/top1000_origin_fact_check_20260901_report.json').read_text())
assert len(audit) == 1_000
assert audit['returned_spotify_id'].is_unique
assert report['rows'] == len(audit)
audit[['popularity_rank', 'monthly_listeners']] = audit[['popularity_rank', 'monthly_listeners']].apply(pd.to_numeric)
total_listeners = audit['monthly_listeners'].sum()
print('Origin audit integrity: PASS')

Origin audit integrity: PASS


## Key findings

In [2]:
summary = (audit.groupby('final_status', as_index=False)
           .agg(bands=('spotify_name', 'size'), monthly_listeners=('monthly_listeners', 'sum')))
summary['band_share'] = summary['bands'] / len(audit)
summary['listener_share'] = summary['monthly_listeners'] / total_listeners
order = ['confirmed', 'corrected', 'resolved', 'contested', 'evidence_for_missing_claim', 'source_variation', 'unresolved']
summary['final_status'] = pd.Categorical(summary['final_status'], order, ordered=True)
summary = summary.sort_values('final_status')
display(summary.style.format({'monthly_listeners': '{:,.0f}', 'band_share': '{:.1%}', 'listener_share': '{:.1%}'}))

fig, ax = plt.subplots(figsize=(9, 4.8))
colors = ['#2f855a', '#c05621', '#805ad5', '#b7791f', '#3182ce', '#d69e2e', '#718096']
ax.barh(summary['final_status'].astype(str), summary['bands'], color=colors)
ax.invert_yaxis()
ax.set(xlabel='Bands', ylabel='', title='Origin fact-check disposition across 1,000 catalogue rows')
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params(axis='y', length=0)
for i, value in enumerate(summary['bands']): ax.text(value + 6, i, f'{value}', va='center')
plt.tight_layout()
plt.show()

,final_status,bands,monthly_listeners,band_share,listener_share
0,confirmed,641,"2,061,045,048",64.1%,89.4%
2,corrected,18,"56,284,557",1.8%,2.4%
4,resolved,11,"11,209,685",1.1%,0.5%
1,contested,2,"7,293,135",0.2%,0.3%
3,evidence_for_missing_claim,187,"91,778,117",18.7%,4.0%
5,source_variation,68,"33,520,419",6.8%,1.5%
6,unresolved,73,"43,149,540",7.3%,1.9%


/var/folders/jk/2s13yhh11zx3vw8rzxvwhcxm0000gn/T/ipykernel_27982/1096859537.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The current audit records 641 confirmed, 18 corrected and 11 resolved origins: 670 rows, representing 92.37% of summed captured monthly-listener counts. All 29 corrections and resolutions already match the current catalogue's origin clusters.

The other 330 rows comprise 187 proposed origins for missing claims, 68 source variations, 73 unresolved records and two contested records. They represent 7.63% of summed monthly-listener counts. A nonblank `final_origin` is not itself a verified or sufficiently precise formation locality: proposals may contain broad country labels, and some unresolved claims remain populated.

These audit statuses differ from geographic coverage. The catalogue contains 749 nonblank origin labels; 663 acts have strict FUA assignments and 666 have assignments under the extended mapping. Do not use these quantities interchangeably.

In [3]:
decisions = audit[audit['manual_status'].isin(['confirmed', 'corrected', 'resolved', 'contested'])][
    ['popularity_rank', 'spotify_name', 'manual_status', 'current_claim_place', 'final_origin', 'manual_notes', 'manual_source_url']
].sort_values('popularity_rank')
display(decisions)
print('Manual conflict decisions:', len(decisions))
print('Corrected/resolved listener share:', f"{audit.loc[audit['manual_status'].isin(['corrected', 'resolved']), 'monthly_listeners'].sum() / total_listeners:.1%}")

,popularity_rank,spotify_name,manual_status,current_claim_place,final_origin,manual_notes,manual_source_url
1,2,Arctic Monkeys,confirmed,Sheffield,Sheffield,High Green is a Sheffield suburb; the labels a...,https://en.wikipedia.org/wiki/Arctic_Monkeys
4,5,One Direction,confirmed,London,London,Wembley is within London; the labels are compa...,https://en.wikipedia.org/wiki/One_Direction
8,9,Gorillaz,confirmed,London,London,The project was created in London; Essex refle...,https://en.wikipedia.org/wiki/Gorillaz
15,16,Dire Straits,confirmed,Deptford,Deptford,Deptford is the specific London formation place.,https://en.wikipedia.org/wiki/Dire_Straits
29,30,The Outfield,corrected,London,London,The band's official biography describes it as ...,https://theoutfield.com/bio
34,35,Eurythmics,corrected,London,London,The duo formed in London; Sunderland is Dave S...,https://rockhall.com/wp-content/uploads/2024/0...
51,52,Pet Shop Boys,confirmed,London,London,Chelsea is within London; the labels are compa...,https://en.wikipedia.org/wiki/Pet_Shop_Boys
52,53,The Hollies,confirmed,Manchester,Manchester,Manchester was historically in Lancashire; the...,https://en.wikipedia.org/wiki/The_Hollies
57,58,Bronski Beat,confirmed,London,London,Brixton is within London; the labels are compa...,https://en.wikipedia.org/wiki/Bronski_Beat
58,59,Seafret,confirmed,Bridlington,Bridlington,Bridlington is within the East Riding of Yorks...,https://en.wikipedia.org/wiki/Seafret


Manual conflict decisions: 56
Corrected/resolved listener share: 2.9%


## Remaining uncertainty

The shortlist below contains the 68 source variations and 73 unresolved records: 141 rows. It does not include the 187 missing-claim proposals or the two contested records, which also require a decision or stronger evidence. The complete pending or contested set therefore contains 330 rows.

In [4]:
pending = audit[audit['final_status'].isin(['source_variation', 'unresolved'])][
    ['popularity_rank', 'spotify_name', 'current_claim_place', 'musicbrainz_begin_area', 'wikipedia_origin', 'final_status', 'review_priority']
].sort_values(['review_priority', 'popularity_rank'])
print('Rows still requiring research or an editorial convention:', len(pending))
display(pending.head(50))

Rows still requiring research or an editorial convention: 141


,popularity_rank,spotify_name,current_claim_place,musicbrainz_begin_area,wikipedia_origin,final_status,review_priority
270,271,PRESIDENT,,,United Kingdom,unresolved,low
312,313,Academy of Ancient Music,,,,unresolved,low
315,316,La Serenissima,,,,unresolved,low
321,322,Wasia Project,,,England,unresolved,low
349,350,Gunship,,,United Kingdom,unresolved,low
369,370,LMC,,,,unresolved,low
380,381,GPF,,,,unresolved,low
388,389,Precious,,United Kingdom,United Kingdom,unresolved,low
408,409,The Big Push,Brighton,,,unresolved,low
441,442,Gloryhammer,,,United Kingdom,unresolved,low


## Methodology, limitations, and next steps

MusicBrainz's `begin-area` is a structured starting-place field; Wikipedia infobox origins are community-edited claims that may encode a later base or a broad city. Agreement is corroboration, not proof. The full dataset review also found that four band entities link to eight individual-member MusicBrainz records. Those additional records currently contribute no distinct begin-area, but their identities must be validated before they can support group-formation evidence.

The 18 corrections and 11 resolutions are already applied. Next, adjudicate the 187 missing-claim proposals, 68 source variations and two contested cases, and research the 73 unresolved records. Prioritize both audience weight and the effect on small FUA band counts. Do not automatically promote proposals or count unresolved records as verified origins. Studio Killers' proposed “Finland,” for example, is a country extracted from multi-country evidence rather than a formation locality.

Keep precise formation locality separate from city clustering and FUA assignment. Report strict FUA coverage as 66.3% of acts and 90.7% of summed monthly-listener counts; extended coverage is 66.6% and 91.0%, respectively. The origin audit does not establish band eligibility: the full review identifies five verified solo projects still included in the catalogue.

Sources: [MusicBrainz artist area documentation](https://musicbrainz.org/doc/Artist%23Area), [MusicBrainz API search documentation](https://musicbrainz.org/doc/MusicBrainz_API/Search), [MediaWiki revisions API](https://www.mediawiki.org/wiki/API%3ARevisions), the row-level URLs in the audit CSV and manual-decision file, and the [full dataset review](top1000_full_review_20260905.ipynb).